# Notebook for demonstrating and testing MAS capabilities of data retrievemnt

The use of individual workflows of this system is also available to users and developers. 

This journal will demonstrate how to use workflows to retrive data from data lake using text requests.

In [ ]:
import sys

sys.path.append('./../src')

## 0. Demontrating task example

We assume that, due to the implementation, this workflow will most likely be relevant when working with code. Therefore, this log also serves as an additional guide on how to obtain data from a request.

To demonstrate the work, it is recommended to first run the results of the data addition demonstration - during the work of the journal, a test data lake will be created, which will be filled with data with metadata values.

In [ ]:
from data_management.local_datalake_management import LocalDataLake
readed_datalake = LocalDataLake.open("./add_testing_storage")
readed_datalake.get_datalake_info()

In [ ]:
TEST_TASK_1 = "Hi! I need some data captured by fluorescent microscope, where pixel sizes along OX and OY axies are more then 1024 an less then 3096. Can you get it from me?"

In [ ]:
dataset = readed_datalake['fluorescent_microscopy']
dataset.get_metadata_df().head(100)

This example shows how to use an SQL query to get images with a layer resolution from 1025 to 4095 pixels.

In [ ]:
sql_request = 'SELECT * FROM df WHERE number_of_pixels_x > 1024 AND number_of_pixels_x < 4096 AND number_of_pixels_y > 1024 AND number_of_pixels_y < 4096'
df = dataset.get_elements_by_sql_request(sql_request)
df.head()

## 1. Demonstrating data retrieving

In [ ]:
import os
import yaml

prompts_names = ['getting_data_sp.yaml']

with open(os.path.join('../src/prompts_templates/getting_data', prompts_names[0])) as stream:
    get_data_sp = yaml.safe_load(stream)['system_prompt']

get_data_sp

In [ ]:
from scidatamas.data_retrieval_workflow import DataRetrievingFlow

add_flow = DataRetrievingFlow(get_data_sp, readed_datalake)

In [ ]:
from scidatamas.data_retrieval_workflow import DataRetrievingFlow, RetrievingDatalakeDataState

def get_data(user_task: str, datalake:LocalDataLake, system_prompt: str, model: str, provider: str):
    add_data_workflow = DataRetrievingFlow(system_prompt=system_prompt, datalake=datalake, model=model, provider=provider)

    working_state = RetrievingDatalakeDataState()
    working_state['users_task'] = user_task
    wf = add_data_workflow.get_workflow()
    wf = wf.compile()
    working_state = wf.invoke(working_state)
    return working_state

In [ ]:
final_state = get_data(TEST_TASK_1, readed_datalake, get_data_sp, model='mistral-large-latest', provider='mistralai')

Here is a result of workflow usage.

In [ ]:
final_state